# Grover's algorithm

Grover's algorithm is:
* a *quantum search algorithm* in unstructured data
* resorts to *quantum amplitude estimation* 
* finds the correct element after $\mathcal{O}(N)$ steps (where $N$ is the number of elements in the unstructured data)

It can be used to:
* find one or more elements in an unstructured data set 
    * it has a limitation: if the number of solutions to find is greater than 0.3 the size of the data set, then Grover's algorithm does not work properly (this is because the number of times we need to apply the oracle and the diffusion operator is less than one)
* solve SAT problems (satisfiability problems)

It is composed of three components:
1. Hadamard transform (applies the Hadamard gate to all the qubits)
    * this transforms guarantees that, at the beginning, all the elements have the same probability to be found
2. Oracle
    * signals the solution(s) by flipping its phase
3. Diffusion operator
    * amplifies the amplitude of the solution(s)

In [1]:
import pennylane as qml
from pennylane import numpy as np
from math import log2, isclose

In [2]:
def state_to_ket(state, *, tol=1e-10, max_terms=None, binary=True, normalize_check=False):
    """
    Convert a statevector to Dirac ket notation.

    Parameters
    ----------
    state : array-like of complex
        1D array of amplitudes, length = 2^n (typically).
    tol : float
        Amplitudes with |amp| <= tol are omitted.
    max_terms : int or None
        If set, shows only the largest `max_terms` amplitudes by magnitude.
    binary : bool
        If True, basis labels are |00...1>. If False, labels are |k>.
    normalize_check : bool
        If True, raises ValueError if state is not ~normalized.

    Returns
    -------
    str : a Dirac-notation string
    """
    v = np.asarray(state, dtype=complex).flatten()
    if v.ndim != 1:
        raise ValueError("state must be a 1D array-like.")
    dim = v.size
    if dim == 0:
        return "0"

    # Optional normalization check
    if normalize_check:
        norm = float(np.vdot(v, v).real)
        if not isclose(norm, 1.0, rel_tol=1e-9, abs_tol=1e-9):
            raise ValueError(f"State not normalized: ||ψ||^2 = {norm}")

    # Determine qubit count if possible
    n = None
    if dim & (dim - 1) == 0:  # power of two
        n = int(log2(dim))

    # Collect nonzero terms
    terms = [(i, amp) for i, amp in enumerate(v) if abs(amp) > tol]
    if not terms:
        return "0"

    # Optionally keep only largest amplitudes
    if max_terms is not None and max_terms > 0 and len(terms) > max_terms:
        terms.sort(key=lambda x: abs(x[1]), reverse=True)
        terms = terms[:max_terms]
        terms.sort(key=lambda x: x[0])

    def fmt_complex(z):
        # Nicely format complex numbers with small real/imag suppressed.
        r = 0.0 if abs(z.real) < tol else z.real
        im = 0.0 if abs(z.imag) < tol else z.imag
        if im == 0.0:
            return f"{r:.6g}"
        if r == 0.0:
            return f"{im:.6g}j"
        sign = "+" if im >= 0 else "-"
        return f"({r:.6g}{sign}{abs(im):.6g}j)"

    def basis_label(i):
        if binary:
            if n is None:
                return f"|{i}⟩"
            return f"|{format(i, f'0{n}b')}⟩"
        return f"|{i}⟩"

    pieces = []
    for i, amp in terms:
        label = basis_label(i)

        # Handle special cases for amplitude formatting
        if abs(amp - 1) <= tol:
            coeff = ""
            sign = "+"
        elif abs(amp + 1) <= tol:
            coeff = ""
            sign = "-"
        else:
            coeff = fmt_complex(amp)
            sign = "+"

        pieces.append((sign, coeff, label))

    # Build string with proper leading sign
    out = []
    for idx, (sign, coeff, label) in enumerate(pieces):
        if idx == 0:
            if sign == "-":
                out.append("-")
            # if coeff empty, just label; else coeff + label
            out.append(f"{coeff}{label}" if coeff else f"{label}")
        else:
            out.append(f" {sign} ")
            out.append(f"{coeff}{label}" if coeff else f"{label}")

    return "".join(out)


def print_state_ket(state, **kwargs):
    """Convenience: print the ket string."""
    print(state_to_ket(state, **kwargs))


## Grover's algorithm with two searching qubits (using ancilla qubit)

With two searching qubits, we have a total of $2^2 = 4$ elements in the unstructured data set: $00$, $01$, $10$, $11$

Let us first consider the scenario where we only have one solution (e.g. $\ket{01}$)

We set the following parameters:
* *n_qubits*: the number of qubits the algorithm will use
    * set to three, because we are using an ancilla qubit (note that it is possible to perform Grover's algorithm without an ancilla qubit)
* *set_key*: a set composed of the elements we want to search in the unstructured data base
* *dev*: setting the PennyLane device

In [3]:
n_register = 2 #n
n_qubits = n_register + 1
key=[1,1]
dev = qml.device("default.qubit", wires=n_qubits)

Let us now see how each part of Grover's algorithm individually, starting with the Hadamard transform

All qubits start in state $\ket{0}$.
However, for later help identify the solutions, the ancilla qubit is flipped to $\ket{1}$.
Thus, the initial state is as follows: 
$$\ket{\psi_0} = \ket{0}^{\otimes n} \otimes \ket{1}$$

After applying the Hadamard transform, the initial state becomes:
$$\ket{\psi_1} = \dfrac{1}{\sqrt{2^n}} \sum_{x \in \{0,1\}^{n}} \ket{x} \otimes \ket{-}$$

In [4]:
@qml.qnode(dev)
def had_transf(n_qubits):
    qml.X(n_qubits-1)
    for i in range(n_qubits):
        qml.H(i)
    return qml.state()

print(state_to_ket(had_transf(n_qubits)))

0.353553|000⟩ + -0.353553|001⟩ + 0.353553|010⟩ + -0.353553|011⟩ + 0.353553|100⟩ + -0.353553|101⟩ + 0.353553|110⟩ + -0.353553|111⟩


Now, let us focus on the oracle.

For that, let $S \subseteq \{0,1\}^{n}$ be the set of solutions and $x \in \{0,1\}^{n}$ 

The oracle is seen as a function 
\begin{align*}
    f: \{0,1\}^{n} &\rightarrow \{0,1\} \\
    x &\mapsto 
    \begin{cases}
        1 &\text{ if } x \in S \\
        0 &\text{ if } x \not\in S
    \end{cases}
\end{align*}

The action of the oracle in $\ket{\psi_1}$ is to identify the solutions, by flipping their phase:
\begin{align*}
    \ket{\psi_2} = f \ket{\psi_1} 
    &= f \left( \dfrac{1}{\sqrt{2^n}} \sum_{x \in \{0,1\}^{n}} \ket{x} \otimes \ket{-} \right) \\
    &= \left( \dfrac{1}{\sqrt{2^n}} \sum_{y \in S} (-1) \ket{y} \otimes \ket{-}  + \dfrac{1}{\sqrt{2^n}} \sum_{\substack{x \in \{0,1\}^{n}\\ x \not\in S}} \ket{x} \right) \otimes \ket{-} \\
    &= \dfrac{1}{\sqrt{2^n}} \sum_{x \in \{0,1\}^{n}} (-1)^{f(x)} \ket{x} \otimes \ket{-}
\end{align*}

In terms of quantum circuits, this is equivalent to apply a *multi-controlled X* gate, in which the control qubits are dictated by the key and the target qubit is the ancilla qubit

In [5]:
@qml.qnode(dev)
def oracle(key, n_qubits):
    ##Hadamard Transform##
    qml.X(n_qubits-1)
    for i in range(n_qubits):
        qml.H(i)
    ##Hadamard Transform##
    qml.ctrl(qml.X,control=range(n_qubits-1), control_values=key)(n_qubits-1) 
    return qml.state()

print(state_to_ket(oracle(key,n_qubits)))

0.353553|000⟩ + -0.353553|001⟩ + 0.353553|010⟩ + -0.353553|011⟩ + 0.353553|100⟩ + -0.353553|101⟩ + -0.353553|110⟩ + 0.353553|111⟩


Notice that the phase of the state $\ket{01}$ has changed!

At last, we turn our attention to the Diffusion operator.

The goal of the Diffusion operator is to amplify the amplitude of the solution.

The Diffusion operator is defined as follows:
$$\hat{D} = 2\ket{\psi}\bra{\psi} - I$$

where $\ket{\psi} = \dfrac{1}{\sqrt{2^n}} \sum_{x \in \{0,1\}^{n} \ket{x}}$

The Diffusion operator is only applied to the register qubits (hence we forget the ancilla in the following computations).
Thus, let us manipulate $\ket{\psi_2}$ in order to separate solution states from the others
\begin{align*}
    &\dfrac{1}{\sqrt{2^n}} \sum_{y \in S} (-1) \ket{y}  + \dfrac{1}{\sqrt{2^n}} \sum_{\substack{x \in \{0,1\}^{n}\\ x \not\in S}} \ket{x} \\
    = 
    &\dfrac{\sqrt{|S|}}{\sqrt{2^n}} \left( \dfrac{\sqrt{2^n}}{\sqrt{|S|}}  \sum_{y \in S} (-1) \ket{y} \right) 
    + \dfrac{\sqrt{2^n - |S|}}{\sqrt{2^n}} \left( \dfrac{\sqrt{2^n}}{\sqrt{2^n - |S|}} \sum_{\substack{x \in \{0,1\}^{n}\\ x \not\in S}} \ket{x} \right) \\
    =
    & \dfrac{\sqrt{|S|}}{\sqrt{2^n}} \ket{s} + \dfrac{\sqrt{2^n - |S|}}{\sqrt{2^n}} \ket{s_{\perp}}
\end{align*}

With a few calculations, the result of applying the Diffusion operator to $\ket{\psi_2}$ is as follows:
\begin{align*}
    \ket{\psi_3} = \hat{D} \ket{\psi_2} 
    &= 
    \hat{D} \left( \dfrac{1}{\sqrt{N}} \sum_{x \in \{0,1\}^{n}} (-1)^{f(x)} \ket{x} \right) \\
    &=
    \hat{D} \left(- \dfrac{\sqrt{M}}{\sqrt{N}} \ket{s} + \dfrac{\sqrt{N - M}}{\sqrt{N}} \ket{s_{\perp}} \right) \\
    &=
    \dfrac{1}{\sqrt{N}} \left( \sqrt{M} + \dfrac{2}{N} \left( \sqrt{(N-1)(N-M)} - \sqrt{M} \right) \right) \ket{s}
    + \dfrac{1}{\sqrt{N}} \left( \dfrac{2}{N} \left( (N-1)\sqrt{N-M}  - \sqrt{(N-1) M} \right) - \sqrt{N-M} \right) \ket{s_{\perp}}
\end{align*}
where $M=|S|$ and $N=2^n$



In [6]:
@qml.qnode(dev)
def diffusion(key, n_qubits):
    ##Hadamard Transform##
    qml.X(n_qubits-1)
    for i in range(n_qubits):
        qml.H(i)
    ##Hadamard Transform##
    ##Oracle##
    qml.ctrl(qml.X,control=range(n_qubits-1), control_values=key)(n_qubits-1)
    ##Oracle##
    for i in range(n_qubits-1):
        qml.H(i)
    for i in range(n_qubits-1):
        qml.X(i)
    qml.ctrl(qml.Z,control=range(n_qubits-2), control_values=[1])(n_qubits-2)
    for i in range(n_qubits-1):
        qml.X(i)
    for i in range(n_qubits-1):
        qml.H(i)
    return qml.state()

print(state_to_ket(diffusion(key,n_qubits)))

-0.707107|110⟩ + 0.707107|111⟩


By putting everything together, we have Grover's algorithm (note that, in this example where we used a register with two qubits, we only needed to apply one time the Oracle and the Diffusion operator)



In [7]:
@qml.qnode(dev)
def grov(key, n_qubits):
    #Hadamard transformation
    qml.X(n_qubits-1)
    for i in range(n_qubits):
        qml.H(i)
    #Oracle
    qml.ctrl(qml.X,control=range(n_qubits-1), control_values=key)(n_qubits-1) 
    #Diffusion operator
    for i in range(n_qubits-1):
        qml.H(i)
    for i in range(n_qubits-1):
        qml.X(i)
    qml.ctrl(qml.Z,control=range(n_qubits-2), control_values=[1])(n_qubits-2)
    for i in range(n_qubits-1):
        qml.X(i)
    for i in range(n_qubits-1):
        qml.H(i)
    return qml.state()

print(state_to_ket(grov(key,n_qubits)))

-0.707107|110⟩ + 0.707107|111⟩


Now, let us implement a general version of Grover's algorithm, in which we can search for $m$ possible solutions

For such task, let us redefine some variables:
* *n_register*: number of register qubits
* *n_qubits*: total number of qubits used in Grover's algorithm (n_register + ancilla_qubit)
* *set_key*: set containing one or more keys to be found

In [8]:
n_register = 3
n_qubits = n_register + 1
set_key = [[0,0,0],[1,1,1]]
dev = qml.device("default.qubit",n_qubits)

def had_transf(n_qubits):
    qml.X(n_qubits-1)
    for i in range(n_qubits):
        qml.H(i)

def oracle(set_key, n_qubits):
    for key in set_key:
        qml.ctrl(qml.X,control=range(n_qubits-1), control_values=key)(n_qubits-1)

def diffusion(n_qubits):
    for i in range(n_qubits-1):
        qml.H(i)
    for i in range(n_qubits-1):
        qml.X(i)
    qml.ctrl(qml.Z,control=range(n_qubits-2), control_values=[1])(n_qubits-2)
    for i in range(n_qubits-1):
        qml.X(i)
    for i in range(n_qubits-1):
        qml.H(i)

@qml.qnode(dev)
def grov(set_key, n_qubits):
    had_transf(n_qubits)
    N = 2**(n_register)
    M = len(set_key)
    aux = np.pi/4 * np.sqrt(N/M) - 0.5
    rep = aux.astype(int)
    for _ in range(rep):
        oracle(set_key, n_qubits)
        diffusion(n_qubits)
    return qml.probs(wires=range(n_qubits-1))

In [9]:
print(qml.draw(grov)(set_key,n_qubits))
print(grov(set_key,n_qubits))

0: ──H────╭○─╭●──H──X─╭●──X──H─┤ ╭Probs
1: ──H────├○─├●──H──X─├●──X──H─┤ ├Probs
2: ──H────├○─├●──H──X─╰Z──X──H─┤ ╰Probs
3: ──X──H─╰X─╰X────────────────┤       
[0.5 0.  0.  0.  0.  0.  0.  0.5]


Now that we have defined a generic version of Grover's algorithm, let us analyze it through another perspective.

As we saw previously, we can split the quantum state into *solution* and *no-solution*:
$$\ket{\psi} = \dfrac{\sqrt{M}}{\sqrt{N}} \ket{s} + \dfrac{\sqrt{N - M}}{\sqrt{N}} \ket{s_{\perp}}$$

where:
\begin{align*}
    \ket{s} = \dfrac{1}{\sqrt{M}} \sum_{y \in S} \ket{y} 
    \qquad
    \ket{s_{\perp}} = \dfrac{1}{\sqrt{M}} \sum_{\substack{x \in \{0,1\}^n \\ x \not\in S}} \ket{x} 
\end{align*}

With some quick calculations we can check the following:
1. $\braket{s | s_{\perp}} = 0 $
    * the vectors $\ket{s}$ and $\ket{s_{\perp}}$ are orthogonal and linearly independent
2. $\braket{s|s} = 1$ and $\braket{s_{\perp}|s_{\perp}} = 1$
    * both vectors are normalized

This entails that the set $\{\ket{s}, \ket{s_{\perp}}\}$ forms a basis of a $2D$-subspace.

Furthermore, any normalized vector in such a subspace can be parametrized by a single angle relative to the basis.
This allows us to rewrite $\ket{\psi}$ w.r.t. to such single angle:
$$\ket{\psi(\theta)} = \sin{(\theta)} \ket{s} + \cos{(\theta)} \ket{s_{\perp}}$$

Having said this, let us also mention something regarding the Oracle.
The one used previously uses an ancilla qubit, in order to kick back the phase, with the Diffusion operator, and amplify the amplitude of the solutions' states.
However, we saw that the Oracle flipped the phase of the solutions, i.e.:
\begin{align*}
    \hat{O} \ket{x} = 
    \begin{cases}
        - \ket{x} & \text{ if } x \in S \\
        \ket{x} & \text{ if } x \not\in S \\
    \end{cases}
\end{align*}

This Oracle can be defined as follows:
$$\hat{O} = I - 2 \ket{s}\bra{s}$$

We can now check the effect that the Oracle has in $\ket{\psi(\theta)}$
$$\hat{O} (\sin{(\theta)} \ket{s} + \cos{(\theta)} \ket{s_{\perp}})  = \sin{(-\theta)} \ket{s} + \cos{(\theta)} \ket{s_{\perp}}$$


As it is possible to see, the Oracle reflects the angle.

Now let us see the effect of the Diffusion operator on the previous state
$$\hat{D} (\sin{(-\theta)} \ket{s} + \cos{(\theta)} \ket{s_{\perp}}) = \sin{(3\theta)} \ket{s} + \cos{(3\theta)} \ket{s_{\perp}}$$

Here, the Diffusion operator does another reflection at the same time it approximates the angle to the $\ket{s}$-axis

Thus, the application of the Oracle and the Diffusion operator adds $2\theta$ to the original angle $\theta$

There are situations in which after applying the Oracle and the Diffusion operator the probability of the solutions to find is not $1$.
Hence we need to apply these two operations more than once.
To see what the optimal number is, we note that $sin(\alpha) = 1$ whenever $\alpha = \dfrac{\pi}{2}$.
Furthermore, applying $k$ times both operators gives on $\ket{\psi}$ gives:
$$\ket{\psi(\theta,k)} = \sin{(2k+1)\theta}\ket{s} + \cos{(2k+1)\theta}\ket{s_{\perp}}$$

By noting that $sin(\theta) = \sqrt{\dfrac{M}{N}}$ and for $N>>1$ we have $\sin{(\theta)} \approx \theta \approx \sqrt{\dfrac{M}{N}}$, then 
\begin{align*}
    & (2k+1)\theta = \dfrac{\pi}{2} \\
    \Leftrightarrow &
    (2k+1) \sqrt{\dfrac{M}{N}} = \dfrac{\pi}{2} \\
    \Leftrightarrow &
    k = \dfrac{\pi}{4} \sqrt{\dfrac{N}{M}} - 0.5
\end{align*}

Hence, to find the solutions, we need to apply around $\dfrac{\pi}{4} \sqrt{\dfrac{N}{M}} - 0.5$ times the Oracle and Diffusion operators.

This brings up two questions:
1. What happens if we apply the Oracle and Diffusion operators more times than indicated?
    * Since *sine* is a periodic function, whenever we apply these operators more than indicated, we distance ourselves from the solution
    * However, there will be a time, due to the fact that *sine* is periodic, that we will get closer to the solution
2. Is there a maximum number of solutions such that Grover's algorithm becomes unusable?
    * Yes, there is. That number is around $0.27$ times the size of the data set $\left( M \leq \dfrac{\pi^2}{36} N \right)$ 

Let us consider a scenario where we explore the maximum number of solutions Grover's algorithm can be useful.

Let us consider a register with 3 qubits.
Hence $M ≈ 0.27 \times 2^3 ≈ 2$

In [10]:
n_register = 3
set_key1 = [[0,0,0],[1,1,1]] #maximum set of solutions where Grover's is useful
set_key2 = [[0,0,0],[0,1,0],[1,1,1]] #set of solutions where Grover's is no longer useful

print(grov(set_key1,n_qubits))
print(grov(set_key2,n_qubits))

[0.5 0.  0.  0.  0.  0.  0.  0.5]
[0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125]


In [11]:
print(grov([[1,1,1]],n_qubits))

[0.03125 0.03125 0.03125 0.03125 0.03125 0.03125 0.03125 0.78125]
